# HafenCity NM5 Full Simulation

This notebook inspects the prepared HafenCity `nm5_full` inputs first, then submits a real COUP-noise task to the local NoiseModelling 5 service.

By default it uses a runtime-first preview package at `downloads/hafencity_full/citypyo/hafencity_half_10m`, which keeps the half AOI but downsamples DEM points to 10 m spacing. Change `COMPLEXITY_PRESET` in the first code cell to move between `preview`, `smoke`, `fast`, `balanced`, `detailed`, and `full_area`. Use `COMPLEXITY_OVERRIDES` there for one-off changes. The notebook writes an override compose file under `outputs/` and bind-mounts `downloads/` into the Docker services so the large prepared data does not need to be copied into the image.

In [16]:
from pathlib import Path
import base64
import copy
import csv
import html
import json
import os
import subprocess
import time
from collections import Counter
from urllib import error, request

ROOT = Path.cwd()
if not (ROOT / "docker-compose.yml").exists():
    ROOT = ROOT.parent

# Change this one value to adjust runtime/quality.
# Options: preview, smoke, fast, balanced, detailed, full_area
COMPLEXITY_PRESET = "balanced"

# Optional one-off edits. Examples:
# COMPLEXITY_OVERRIDES = {"max_speed": 30}
# COMPLEXITY_OVERRIDES = {"nm5_settings": {"reflection_order": 1, "max_area": 1000}}
COMPLEXITY_OVERRIDES = {}

# Local runtime tuning. Docker worker CPU limits still matter, so raising
# thread_number only helps if the worker container also gets more CPU.
HOST_LOGICAL_CPUS = os.cpu_count() or 4
LOCAL_WORKER_CPUS = int(os.getenv("NM5_LOCAL_WORKER_CPUS", min(10, max(1, HOST_LOGICAL_CPUS - 2))))
LOCAL_NM5_THREADS = int(os.getenv("NM5_LOCAL_THREADS", max(1, LOCAL_WORKER_CPUS - 1)))
RUN_SINGLE_WORKER = True

BASE_NM5_SETTINGS = {
    "receiver_height": 4,
    "road_width": 1.5,
    "thread_number": LOCAL_NM5_THREADS,
    "iso_classes": "45,50,55,60,65,70,75,200",
}

COMPLEXITY_PRESETS = {
    "preview": {
        "description": "Runtime-first half-AOI preview; very coarse mesh and short propagation distance.",
        "city_pyo_user": "hafencity_half_10m",
        "output_stem": "hafencity_half_10m_nm5_full_preview",
        "max_speed": 50,
        "traffic_quota": 1.0,
        "wall_absorption": 0.69,
        "poll_seconds": 5,
        "timeout_seconds": 1800,
        "nm5_settings": {
            **BASE_NM5_SETTINGS,
            "max_cell_dist": 750,
            "max_area": 25000,
            "reflection_order": 0,
            "max_source_distance": 150,
            "max_reflection_distance": 25,
            "diff_vertical": False,
            "diff_horizontal": False,
            "max_error": 1.0,
        },
    },
    "smoke": {
        "description": "Quick half-AOI check; coarse receiver mesh; no reflections/diffraction.",
        "city_pyo_user": "hafencity_half_10m",
        "output_stem": "hafencity_half_10m_nm5_full",
        "max_speed": 50,
        "traffic_quota": 1.0,
        "wall_absorption": 0.69,
        "poll_seconds": 10,
        "timeout_seconds": 7200,
        "nm5_settings": {
            **BASE_NM5_SETTINGS,
            "max_cell_dist": 150,
            "max_area": 5000,
            "reflection_order": 0,
            "max_source_distance": 250,
            "max_reflection_distance": 50,
            "diff_vertical": False,
            "diff_horizontal": False,
        },
    },
    "fast": {
        "description": "Half-AOI check with a less coarse mesh and longer propagation distance.",
        "city_pyo_user": "hafencity_half_10m",
        "max_speed": 50,
        "traffic_quota": 1.0,
        "wall_absorption": 0.69,
        "poll_seconds": 10,
        "timeout_seconds": 10800,
        "nm5_settings": {
            **BASE_NM5_SETTINGS,
            "max_cell_dist": 250,
            "max_area": 2500,
            "reflection_order": 0,
            "max_source_distance": 400,
            "max_reflection_distance": 50,
            "diff_vertical": False,
            "diff_horizontal": False,
        },
    },
    "balanced": {
        "description": "Half-AOI run with a finer mesh and horizontal diffraction enabled.",
        "city_pyo_user": "hafencity_half_10m",
        "max_speed": 50,
        "traffic_quota": 1.0,
        "wall_absorption": 0.69,
        "poll_seconds": 15,
        "timeout_seconds": 21600,
        "nm5_settings": {
            **BASE_NM5_SETTINGS,
            "max_cell_dist": 500,
            "max_area": 1000,
            "reflection_order": 0,
            "max_source_distance": 600,
            "max_reflection_distance": 100,
            "diff_vertical": False,
            "diff_horizontal": True,
        },
    },
    "detailed": {
        "description": "Half-AOI detailed run; finer mesh, one reflection, vertical/horizontal diffraction.",
        "city_pyo_user": "hafencity_half_10m",
        "max_speed": 50,
        "traffic_quota": 1.0,
        "wall_absorption": 0.69,
        "poll_seconds": 30,
        "timeout_seconds": 43200,
        "nm5_settings": {
            **BASE_NM5_SETTINGS,
            "max_cell_dist": 750,
            "max_area": 275,
            "reflection_order": 1,
            "max_source_distance": 750,
            "max_reflection_distance": 350,
            "diff_vertical": True,
            "diff_horizontal": True,
        },
    },
    "full_area": {
        "description": "Complete HafenCity package; use for final checks, expect a much longer run.",
        "city_pyo_user": "hafencity_full",
        "max_speed": 50,
        "traffic_quota": 1.0,
        "wall_absorption": 0.69,
        "poll_seconds": 60,
        "timeout_seconds": 86400,
        "nm5_settings": {
            **BASE_NM5_SETTINGS,
            "max_cell_dist": 500,
            "max_area": 1000,
            "reflection_order": 0,
            "max_source_distance": 600,
            "max_reflection_distance": 100,
            "diff_vertical": False,
            "diff_horizontal": True,
        },
    },
}

def deep_update(base, override):
    result = copy.deepcopy(base)
    for key, value in override.items():
        if isinstance(value, dict) and isinstance(result.get(key), dict):
            result[key] = deep_update(result[key], value)
        else:
            result[key] = value
    return result

if COMPLEXITY_PRESET not in COMPLEXITY_PRESETS:
    raise ValueError(f"Unknown COMPLEXITY_PRESET: {COMPLEXITY_PRESET}")

RUN_CONFIG = deep_update(COMPLEXITY_PRESETS[COMPLEXITY_PRESET], COMPLEXITY_OVERRIDES)
CITY_PYO_USER = RUN_CONFIG["city_pyo_user"]
OUTPUT_STEM = RUN_CONFIG.get("output_stem") or f"{CITY_PYO_USER}_nm5_full_{COMPLEXITY_PRESET}"
PACKAGE = ROOT / "downloads" / "hafencity_full" / "citypyo" / CITY_PYO_USER
PROCESSED = ROOT / "downloads" / "hafencity_full" / "processed"
OUTPUTS = ROOT / "outputs"
OUTPUTS.mkdir(exist_ok=True)

API_URL = "http://localhost:5001"
AUTH_USER = "dev"
AUTH_PASSWORD = "dev"
print("complexity preset:", COMPLEXITY_PRESET)
print("preset description:", RUN_CONFIG["description"])
print("host logical CPUs:", HOST_LOGICAL_CPUS)
print("local worker CPU limit:", LOCAL_WORKER_CPUS)
print("NM5 thread_number:", RUN_CONFIG["nm5_settings"].get("thread_number"))
print("repo root:", ROOT)
print("local package:", PACKAGE)
print("package exists:", PACKAGE.exists())
print("output stem:", OUTPUT_STEM)

complexity preset: balanced
preset description: Half-AOI run with a finer mesh and horizontal diffraction enabled.
host logical CPUs: 22
local worker CPU limit: 10
NM5 thread_number: 9
repo root: c:\Users\dmz-admin\CodeProjects\coupnoise\COUP-noise
local package: c:\Users\dmz-admin\CodeProjects\coupnoise\COUP-noise\downloads\hafencity_full\citypyo\hafencity_half_10m
package exists: True
output stem: hafencity_half_10m_nm5_full_balanced


## Display Helpers

The plotting helpers below use plain SVG/HTML so the notebook does not require `geopandas`, `folium`, or `matplotlib` in the host environment.

In [17]:
try:
    from IPython.display import HTML, display
except Exception:
    HTML = None
    def display(value):
        print(value)

def load_json(path):
    with Path(path).open(encoding="utf-8") as handle:
        return json.load(handle)

def read_csv_rows(path):
    with Path(path).open(encoding="utf-8", newline="") as handle:
        return list(csv.DictReader(handle))

def iter_positions(coords):
    if not coords:
        return
    if isinstance(coords[0], (int, float)):
        yield float(coords[0]), float(coords[1])
        return
    for child in coords:
        yield from iter_positions(child)

def feature_collection_bbox(payload):
    xs, ys = [], []
    for feature in payload.get("features", []):
        geometry = feature.get("geometry") or {}
        for x, y in iter_positions(geometry.get("coordinates")):
            xs.append(x)
            ys.append(y)
    return (min(xs), min(ys), max(xs), max(ys)) if xs else None

def html_table(rows, max_rows=25):
    rows = list(rows)
    if not rows:
        return HTML("<p>No rows.</p>") if HTML else "No rows."
    columns = list(rows[0].keys())
    body = []
    body.append("<table style='border-collapse:collapse;font:13px sans-serif'>")
    body.append("<thead><tr>" + "".join(f"<th style='border:1px solid #ccc;padding:4px 6px;text-align:left'>{html.escape(str(c))}</th>" for c in columns) + "</tr></thead>")
    body.append("<tbody>")
    for row in rows[:max_rows]:
        body.append("<tr>" + "".join(f"<td style='border:1px solid #ddd;padding:4px 6px'>{html.escape(str(row.get(c, '')))}</td>" for c in columns) + "</tr>")
    body.append("</tbody></table>")
    if len(rows) > max_rows:
        body.append(f"<p>Showing {max_rows} of {len(rows)} rows.</p>")
    return HTML("\n".join(body)) if HTML else "\n".join(str(row) for row in rows[:max_rows])

def geometry_svg(geometry, sx, sy, style):
    geometry_type = geometry.get("type")
    coordinates = geometry.get("coordinates")
    parts = []
    if geometry_type == "Point":
        x, y = coordinates[:2]
        parts.append(f"<circle cx='{sx(x):.2f}' cy='{sy(y):.2f}' r='1.5' {style}/>")
    elif geometry_type == "MultiPoint":
        for x, y, *_ in coordinates:
            parts.append(f"<circle cx='{sx(x):.2f}' cy='{sy(y):.2f}' r='1.1' {style}/>")
    elif geometry_type == "LineString":
        points = " ".join(f"{sx(x):.2f},{sy(y):.2f}" for x, y, *_ in coordinates)
        parts.append(f"<polyline points='{points}' {style}/>")
    elif geometry_type == "MultiLineString":
        for line in coordinates:
            points = " ".join(f"{sx(x):.2f},{sy(y):.2f}" for x, y, *_ in line)
            parts.append(f"<polyline points='{points}' {style}/>")
    elif geometry_type == "Polygon":
        for ring_index, ring in enumerate(coordinates):
            points = " ".join(f"{sx(x):.2f},{sy(y):.2f}" for x, y, *_ in ring)
            parts.append(f"<polygon points='{points}' {style}/>")
    elif geometry_type == "MultiPolygon":
        for polygon in coordinates:
            for ring in polygon:
                points = " ".join(f"{sx(x):.2f},{sy(y):.2f}" for x, y, *_ in ring)
                parts.append(f"<polygon points='{points}' {style}/>")
    return "\n".join(parts)

def layer_style(feature, layer_name):
    props = feature.get("properties") or {}
    if layer_name == "roads":
        if props.get("road_type") == "railroad":
            return "fill='none' stroke='#7b2cbf' stroke-width='1.6' stroke-opacity='0.9'"
        color = "#d9480f" if props.get("traffic_count_source") else "#f08c00"
        return f"fill='none' stroke='{color}' stroke-width='1.2' stroke-opacity='0.85'"
    if layer_name == "ground":
        g = float(props.get("G", 0.5))
        if g <= 0.05:
            fill = "#748ffc"
        elif g < 0.3:
            fill = "#adb5bd"
        elif g < 0.8:
            fill = "#fcc419"
        else:
            fill = "#51cf66"
        return f"fill='{fill}' fill-opacity='0.45' stroke='{fill}' stroke-width='0.5' stroke-opacity='0.8'"
    if layer_name == "dem":
        return "fill='#1971c2' fill-opacity='0.35' stroke='none'"
    if layer_name == "result":
        colors = {0:'#e8f5e9',1:'#c8e6c9',2:'#fff9c4',3:'#ffe082',4:'#ffb74d',5:'#ff8a65',6:'#ef5350',7:'#b71c1c'}
        color = colors.get(int(props.get("idiso", 0) or 0), '#777')
        return f"fill='{color}' fill-opacity='0.55' stroke='{color}' stroke-width='0.8' stroke-opacity='0.9'"
    if layer_name == "aoi":
        return "fill='#4dabf7' fill-opacity='0.12' stroke='#1864ab' stroke-width='2'"
    return "fill='#495057' fill-opacity='0.5' stroke='#212529' stroke-width='0.5'"

def show_geojson(payload, layer_name, title, max_features=None, bbox=None, width=900, height=520, crs_label="EPSG:25832"):
    features = payload.get("features", [])
    if max_features and len(features) > max_features:
        stride = max(1, len(features) // max_features)
        features = features[::stride][:max_features]
    draw_payload = {"type": "FeatureCollection", "features": features}
    bbox = bbox or feature_collection_bbox(draw_payload)
    if not bbox:
        display(HTML(f"<p>{html.escape(title)}: no geometry.</p>") if HTML else f"{title}: no geometry")
        return
    min_x, min_y, max_x, max_y = bbox
    coord_format = "{:.6f}" if crs_label == "EPSG:4326" else "{:.1f}"
    bbox_text = ", ".join(coord_format.format(value) for value in (min_x, min_y, max_x, max_y))
    min_pad = 1e-5 if crs_label == "EPSG:4326" else 1
    pad_x = max((max_x - min_x) * 0.04, min_pad)
    pad_y = max((max_y - min_y) * 0.04, min_pad)
    min_x -= pad_x; max_x += pad_x; min_y -= pad_y; max_y += pad_y
    def sx(x):
        return 18 + (float(x) - min_x) / (max_x - min_x) * (width - 36)
    def sy(y):
        return height - 18 - (float(y) - min_y) / (max_y - min_y) * (height - 36)
    shapes = []
    for feature in features:
        shapes.append(geometry_svg(feature.get("geometry") or {}, sx, sy, layer_style(feature, layer_name)))
    svg = f"""
    <div style='font:14px sans-serif'>
      <h3 style='margin:0 0 6px'>{html.escape(title)}</h3>
      <div style='color:#555;margin-bottom:8px'>{len(features)} displayed feature(s), bbox {html.escape(crs_label)}: {bbox_text}</div>
      <svg width='{width}' height='{height}' style='border:1px solid #ccc;background:#f8f9fa'>
        {''.join(shapes)}
      </svg>
    </div>
    """
    display(HTML(svg) if HTML else svg)


## Input Inventory

In [18]:
layer_files = {
    "project_area": PACKAGE / "project_area.geojson",
    "buildings_upperfloor": PACKAGE / "upperfloor.geojson",
    "roads_and_rail": PACKAGE / "roads.geojson",
    "dem": PACKAGE / "dem.geojson",
    "ground_absorption": PACKAGE / "ground_absorption.geojson",
    "atmospheric_settings": PACKAGE / "atmospheric_settings.csv",
    "traffic_counts_reference": PACKAGE / "traffic_counts_reference.geojson",
    "alkis_reference": PACKAGE / "alkis_reference.geojson",
    "traffic_assignments": PROCESSED / "traffic_count_assignments_hafencity.csv",
    "gtfs_rail_assignments": PROCESSED / "gtfs_rail_assignments_hafencity.csv",
}

inventory = []
for name, path in layer_files.items():
    item = {"layer": name, "path": str(path.relative_to(ROOT)), "exists": path.exists(), "bytes": path.stat().st_size if path.exists() else 0}
    if path.exists() and path.suffix.lower() == ".geojson":
        item["features"] = len(load_json(path).get("features", []))
    elif path.exists() and path.suffix.lower() == ".csv":
        item["rows"] = len(read_csv_rows(path))
    inventory.append(item)

display(html_table(inventory, max_rows=30))

layer,path,exists,bytes,features
project_area,downloads\hafencity_full\citypyo\hafencity_half_10m\project_area.geojson,True,461,1
buildings_upperfloor,downloads\hafencity_full\citypyo\hafencity_half_10m\upperfloor.geojson,True,38922,56
roads_and_rail,downloads\hafencity_full\citypyo\hafencity_half_10m\roads.geojson,True,97294,167
dem,downloads\hafencity_full\citypyo\hafencity_half_10m\dem.geojson,True,361119,2880
ground_absorption,downloads\hafencity_full\citypyo\hafencity_half_10m\ground_absorption.geojson,True,143339,142
atmospheric_settings,downloads\hafencity_full\citypyo\hafencity_half_10m\atmospheric_settings.csv,True,768,
traffic_counts_reference,downloads\hafencity_full\citypyo\hafencity_half_10m\traffic_counts_reference.geojson,True,3911,4
alkis_reference,downloads\hafencity_full\citypyo\hafencity_half_10m\alkis_reference.geojson,True,1354757,2195
traffic_assignments,downloads\hafencity_full\processed\traffic_count_assignments_hafencity.csv,True,481,
gtfs_rail_assignments,downloads\hafencity_full\processed\gtfs_rail_assignments_hafencity.csv,True,13183,


## Project Area

In [19]:
project_area = load_json(PACKAGE / "project_area.geojson")
AOI_BBOX = feature_collection_bbox(project_area)
show_geojson(project_area, "aoi", "AOI / project_area.geojson")

## Buildings: LoD2-Derived `upperfloor.geojson`

In [20]:
buildings = load_json(PACKAGE / "upperfloor.geojson")
heights = [float((feature.get("properties") or {}).get("HEIGHT", 0)) for feature in buildings.get("features", [])]
print("building features:", len(heights))
print("height min/max/avg:", min(heights), max(heights), round(sum(heights) / len(heights), 2))
show_geojson(buildings, "buildings", "LoD2 buildings / upperfloor.geojson", bbox=AOI_BBOX)

building features: 56
height min/max/avg: 5.773 54.456 27.17


## Roads And Rail: OSM + Traffic Counts + GTFS

In [21]:
roads = load_json(PACKAGE / "roads.geojson")
road_features = [feature for feature in roads.get("features", []) if (feature.get("properties") or {}).get("road_type") != "railroad"]
rail_features = [feature for feature in roads.get("features", []) if (feature.get("properties") or {}).get("road_type") == "railroad"]
traffic_assigned = [feature for feature in road_features if (feature.get("properties") or {}).get("traffic_count_source")]
gtfs_assigned = [feature for feature in rail_features if (feature.get("properties") or {}).get("gtfs_source")]
print("road features:", len(road_features))
print("rail features:", len(rail_features))
print("traffic-count-enriched roads:", len(traffic_assigned))
print("GTFS-enriched rail features:", len(gtfs_assigned))
print("road classes:", Counter((feature.get("properties") or {}).get("fclass") for feature in road_features))
print("rail classes:", Counter((feature.get("properties") or {}).get("fclass") for feature in rail_features))
show_geojson(roads, "roads", "Roads and rail / roads.geojson", bbox=AOI_BBOX)

road features: 161
rail features: 6
traffic-count-enriched roads: 4
GTFS-enriched rail features: 6
road classes: Counter({'secondary': 61, 'service': 39, 'unclassified': 28, 'residential': 17, 'primary': 13, 'secondary_link': 3})
rail classes: Counter({'subway': 6})


## DEM Terrain Points

In [22]:
dem = load_json(PACKAGE / "dem.geojson")
z_values = [float(feature["geometry"]["coordinates"][2]) for feature in dem.get("features", []) if len(feature.get("geometry", {}).get("coordinates", [])) >= 3]
print("DEM points:", len(z_values))
print("elevation min/max/avg:", min(z_values), max(z_values), round(sum(z_values) / len(z_values), 3))
show_geojson(dem, "dem", "DEM terrain points / dem.geojson (sampled)", max_features=6000, bbox=AOI_BBOX)

DEM points: 2880
elevation min/max/avg: 2.0 14.68 5.702


## Ground Absorption: ALKIS `G` Polygons

In [23]:
ground = load_json(PACKAGE / "ground_absorption.geojson")
g_counter = Counter(float((feature.get("properties") or {}).get("G", 0)) for feature in ground.get("features", []))
print("ground polygons:", len(ground.get("features", [])))
print("G distribution:", dict(sorted(g_counter.items())))
show_geojson(ground, "ground", "ALKIS-derived ground absorption / ground_absorption.geojson", bbox=AOI_BBOX)

ground polygons: 142
G distribution: {0.0: 83, 0.05: 30, 0.1: 9, 0.2: 7, 0.25: 10, 0.7: 1, 0.9: 2}


## Atmospheric Settings

In [24]:
atmospheric_rows = read_csv_rows(PACKAGE / "atmospheric_settings.csv")
display(html_table(atmospheric_rows, max_rows=10))

PERIOD,TEMPERATURE,PRESSURE,HUMIDITY,GDISC,PRIME2520,WINDROSE_0,WINDROSE_1,WINDROSE_2,WINDROSE_3,WINDROSE_4,WINDROSE_5,WINDROSE_6,WINDROSE_7,WINDROSE_8,WINDROSE_9,WINDROSE_10,WINDROSE_11,WINDROSE_12,WINDROSE_13,WINDROSE_14,WINDROSE_15
D,9.76,101596.5,76.1,true,false,0.017799,0.014704,0.028788,0.040396,0.088686,0.054171,0.057886,0.069649,0.048135,0.074137,0.122891,0.100449,0.111747,0.095651,0.058814,0.016097
E,8.56,101621.3,81.51,true,false,0.019337,0.020258,0.058471,0.05709,0.089319,0.050645,0.048343,0.064917,0.046501,0.060313,0.084254,0.076888,0.105433,0.115101,0.077808,0.025322
N,6.54,101605.5,89.21,true,false,0.026195,0.018382,0.036075,0.061121,0.099265,0.056985,0.058364,0.077895,0.033778,0.067325,0.117188,0.086397,0.091682,0.054688,0.077895,0.036765


## Traffic Count Assignments

In [25]:
traffic_assignment_rows = read_csv_rows(PROCESSED / "traffic_count_assignments_hafencity.csv")
display(html_table(traffic_assignment_rows, max_rows=20))

traffic_count_station,traffic_count_name,road_pk,road_name,road_class,distance_m,light_traffic_daily,heavy_traffic_daily,heavy_share_percent
111940,Deichtortunnel O Meßberg,225,Willy-Brandt-Straße,primary,0.11,50445,2655,5.0
110970,Oberbaumbrücke SW Deichtorplatz,40,Oberbaumbrücke,secondary,0.2,13440,560,4.0
409940,Überseeallee O Osakaallee,213,Überseeallee,secondary,5.12,6048,252,4.0
113920,Shanghaiallee N Überseeallee,68,Shanghaiallee,secondary,4.16,9951,749,7.0


## GTFS Rail Assignments

In [26]:
gtfs_assignment_rows = read_csv_rows(PROCESSED / "gtfs_rail_assignments_hafencity.csv")
freqs = [float(row["trains_per_hour"]) for row in gtfs_assignment_rows]
print("GTFS assignment rows:", len(gtfs_assignment_rows))
print("trains/hour min/max/avg:", min(freqs), max(freqs), round(sum(freqs) / len(freqs), 2))
display(html_table(gtfs_assignment_rows, max_rows=12))

GTFS assignment rows: 94
trains/hour min/max/avg: 0.25 47.56 10.81


rail_pk,rail_class,gtfs_shape_id,gtfs_routes,distance_m,trains_per_hour,train_speed,train_type
228,rail,6144;6145;6155;6159;6169;6176;6177;6178;6192;6193;6217;6218;6220;6229;6231;6232;6239;6324;6326;6327;6331;6332;6333;6340;6342;6343;6348;6351;6355;6359;6363,RB31;RB41;RE3;RE4;RE5,13.6,15.81,100,FLIRT-4U1
229,rail,6144,RB31,63.46,0.25,100,FLIRT-4U1
231,rail,6144,RB31,49.59,0.25,100,FLIRT-4U1
232,rail,6144,RB31,85.3,0.25,100,FLIRT-4U1
233,rail,6144;6145;6155;6159;6169;6176;6177;6178;6192;6193;6217;6218;6220;6229;6231;6232;6239;6324;6326;6327;6331;6332;6333;6340;6342;6343;6348;6351;6355;6359;6363,RB31;RB41;RE3;RE4;RE5,15.37,15.81,100,FLIRT-4U1
235,rail,6144;6145;6155;6159;6169;6176;6177;6178;6192;6193;6217;6218;6220;6229;6231;6232;6239;6324;6326;6327;6331;6332;6333;6340;6342;6343;6348;6351;6355;6359;6363,RB31;RB41;RE3;RE4;RE5,14.89,15.81,100,FLIRT-4U1
236,subway,4213;4216;4220;4227;4229;4231;4235;4236;4241;4243;4245;4247;4268;4269;4270;4272;4275;4276;4277;4295;4296;4297;4298;4299;4300;4301;4302;4303;4305;4306;4310;4311;4313;4314;4315;4316;4317;4318;4320;4321;4323;4326;4327;4328;4329;4330;4331;4332;4333;4334;4336;4337;4338;4339,U1,0.0,46.4,60,Z50000-7U1
237,rail,6144;6145;6155;6159;6169;6176;6177;6178;6192;6193;6217;6218;6220;6229;6231;6232;6239;6324;6326;6327;6331;6332;6333;6340;6342;6343;6348;6351;6355;6359;6363,RB31;RB41;RE3;RE4;RE5,9.21,15.81,100,FLIRT-4U1
238,subway,4213;4216;4220;4227;4229;4231;4235;4236;4241;4243;4245;4247;4268;4269;4270;4272;4275;4276;4277;4295;4296;4297;4298;4299;4300;4301;4302;4303;4305;4306;4310;4311;4313;4314;4315;4316;4317;4318;4320;4321;4323;4326;4327;4328;4329;4330;4331;4332;4333;4334;4336;4337;4338;4339,U1,0.0,46.4,60,Z50000-7U1
239,subway,4213;4216;4220;4227;4229;4231;4235;4236;4241;4243;4245;4247;4268;4269;4270;4272;4275;4276;4277;4295;4296;4297;4298;4299;4300;4301;4302;4303;4305;4306;4310;4311;4313;4314;4315;4316;4317;4318;4320;4321;4323;4326;4327;4328;4329;4330;4331;4332;4333;4334;4336;4337;4338;4339,U1,0.0,46.4,60,Z50000-7U1


## Start The `nm5_full` Docker Stack

This cell writes a compose override file in `outputs/` and starts the API/worker services with `CITY_PYO=/app/downloads/hafencity_full/citypyo` and `NOISE_ENGINE=nm5_full`.

Set `START_STACK = False` if the stack is already running with the same settings.

In [27]:
START_STACK = False
COMPOSE_OVERRIDE = OUTPUTS / "docker-compose.hafencity-nm5-full.yml"

override_yaml = f"""
services:
  api:
    environment:
      - CITY_PYO=/app/downloads/hafencity_full/citypyo
      - NOISE_ENGINE=nm5_full
      - CELERY_QUEUE=noise_nm5_full
      - CLIENT_ID=dev
      - CLIENT_PASSWORD=dev
      - REDIS_PASS=devredis
    volumes:
      - \"{(ROOT / 'downloads').resolve().as_posix()}:/app/downloads:ro\"
  worker_1:
    cpus: \"{LOCAL_WORKER_CPUS}\"
    environment:
      - CITY_PYO=/app/downloads/hafencity_full/citypyo
      - NOISE_ENGINE=nm5_full
      - CELERY_QUEUE=noise_nm5_full
      - REDIS_PASS=devredis
      - JAVA_TOOL_OPTIONS=-XX:ActiveProcessorCount={LOCAL_WORKER_CPUS}
    volumes:
      - \"{(ROOT / 'downloads').resolve().as_posix()}:/app/downloads:ro\"
    deploy:
      resources:
        limits:
          cpus: \"{LOCAL_WORKER_CPUS}\"
  worker_2:
    cpus: \"{LOCAL_WORKER_CPUS}\"
    environment:
      - CITY_PYO=/app/downloads/hafencity_full/citypyo
      - NOISE_ENGINE=nm5_full
      - CELERY_QUEUE=noise_nm5_full
      - REDIS_PASS=devredis
      - JAVA_TOOL_OPTIONS=-XX:ActiveProcessorCount={LOCAL_WORKER_CPUS}
    volumes:
      - \"{(ROOT / 'downloads').resolve().as_posix()}:/app/downloads:ro\"
    deploy:
      resources:
        limits:
          cpus: \"{LOCAL_WORKER_CPUS}\"
""".strip() + "\n"

COMPOSE_OVERRIDE.write_text(override_yaml, encoding="utf-8")
print("wrote", COMPOSE_OVERRIDE)

def docker_compose_args(*args):
    attempts = []
    for candidate in (["docker", "compose"], ["docker-compose"]):
        probe = subprocess.run(
            candidate + ["version"],
            cwd=ROOT,
            capture_output=True,
            text=True,
            encoding="utf-8",
            errors="replace",
        )
        if probe.returncode == 0:
            return candidate + [
                "-f", str(ROOT / "docker-compose.yml"),
                "-f", str(COMPOSE_OVERRIDE),
                *args,
            ]
        attempts.append((candidate, probe))
    details = "\n\n".join(
        f"{' '.join(command)} version failed:\n{completed.stderr or completed.stdout}"
        for command, completed in attempts
    )
    raise RuntimeError(f"No working Docker Compose command found.\n{details}")

def run_compose(*args):
    command = docker_compose_args(*args)
    completed = subprocess.run(
        command,
        cwd=ROOT,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
    )
    if completed.stdout:
        print(completed.stdout)
    if completed.returncode != 0:
        if completed.stderr:
            print(completed.stderr)
        raise subprocess.CalledProcessError(
            completed.returncode,
            command,
            output=completed.stdout,
            stderr=completed.stderr,
        )
    if completed.stderr:
        print(completed.stderr)

if START_STACK:
    up_args = ["up", "-d", "--build"]
    if RUN_SINGLE_WORKER:
        up_args.extend(["--scale", "worker_2=0"])
    run_compose(*up_args)
    print("nm5_full stack requested")
    print("single worker mode:", RUN_SINGLE_WORKER)
    print("worker CPU limit:", LOCAL_WORKER_CPUS)

wrote c:\Users\dmz-admin\CodeProjects\coupnoise\COUP-noise\outputs\docker-compose.hafencity-nm5-full.yml


## Submit The Full Noise Simulation

The request below asks for GeoJSON contours. `traffic_quota=1.0` keeps the prepared traffic values. The API still requires `max_speed`; current COUP-noise scenario handling applies it only to adjustable road features.

In [ ]:
def auth_header(user, password):
    token = base64.b64encode(f"{user}:{password}".encode("utf-8")).decode("ascii")
    return {"Authorization": f"Basic {token}"}

def http_json(url, method="GET", payload=None, timeout=60):
    data = None
    headers = auth_header(AUTH_USER, AUTH_PASSWORD)
    if payload is not None:
        data = json.dumps(payload).encode("utf-8")
        headers["Content-Type"] = "application/json"
    req = request.Request(url, data=data, headers=headers, method=method)
    try:
        with request.urlopen(req, timeout=timeout) as response:
            return json.loads(response.read().decode("utf-8"))
    except error.HTTPError as exc:
        body = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"HTTP {exc.code} for {url}: {body}") from exc

def wait_for_api(seconds=180):
    deadline = time.time() + seconds
    last_error = None
    while time.time() < deadline:
        try:
            http_json(f"{API_URL}/tasks/not-a-real-task", timeout=5)
            return
        except Exception as exc:
            last_error = exc
            time.sleep(3)
    raise RuntimeError(f"API did not become reachable: {last_error}")

def submit_task(payload):
    response = http_json(f"{API_URL}/task", method="POST", payload=payload, timeout=60)
    return response["taskId"]

def poll_task(task_id, poll_seconds=5, timeout_seconds=7200):
    deadline = time.time() + timeout_seconds
    while time.time() < deadline:
        payload = http_json(f"{API_URL}/tasks/{task_id}", timeout=60)
        state = payload.get("taskState")
        print(time.strftime("%H:%M:%S"), state, "ready=", payload.get("resultReady"))
        if payload.get("resultReady"):
            if state == "FAILURE":
                raise RuntimeError(payload.get("result"))
            return payload["result"]
        time.sleep(poll_seconds)
    raise TimeoutError(f"Task {task_id} did not finish within {timeout_seconds} seconds")

TASK_PAYLOAD = {
    "city_pyo_user": CITY_PYO_USER,
    "result_format": RUN_CONFIG.get("result_format", "geojson"),
    "noise_engine": "nm5_full",
    "max_speed": RUN_CONFIG["max_speed"],
    "traffic_quota": RUN_CONFIG["traffic_quota"],
    "wall_absorption": RUN_CONFIG["wall_absorption"],
    "nm5_settings": RUN_CONFIG["nm5_settings"],
}

print("complexity preset:", COMPLEXITY_PRESET)
print("submitting CityPyo user:", TASK_PAYLOAD["city_pyo_user"])
print("output stem:", OUTPUT_STEM)
print("submitting NM5 settings:")
print(json.dumps(TASK_PAYLOAD["nm5_settings"], indent=2))

wait_for_api()
task_id = submit_task(TASK_PAYLOAD)
print("submitted task:", task_id)
result_geojson = poll_task(
    task_id,
    poll_seconds=RUN_CONFIG.get("poll_seconds", 10),
    timeout_seconds=RUN_CONFIG.get("timeout_seconds", 7200),
)

result_path = OUTPUTS / f"{OUTPUT_STEM}.geojson"
result_path.write_text(json.dumps(result_geojson, indent=2), encoding="utf-8")
print("saved", result_path)

try:
    import sys
    if str(ROOT) not in sys.path:
        sys.path.insert(0, str(ROOT))
    from tools.save_noise_result import _build_map_html

    map_path = OUTPUTS / f"{OUTPUT_STEM}.map.html"
    map_path.write_text(_build_map_html(f"{OUTPUT_STEM} ({COMPLEXITY_PRESET})", result_geojson), encoding="utf-8")
    print("saved", map_path)
except Exception as exc:
    print("map html not written:", exc)


complexity preset: balanced
submitting CityPyo user: hafencity_half_10m
output stem: hafencity_half_10m_nm5_full_balanced
submitting NM5 settings:
{
  "receiver_height": 4,
  "road_width": 1.5,
  "thread_number": 9,
  "iso_classes": "45,50,55,60,65,70,75,200",
  "max_cell_dist": 500,
  "max_area": 1000,
  "reflection_order": 0,
  "max_source_distance": 600,
  "max_reflection_distance": 100,
  "diff_vertical": false,
  "diff_horizontal": true
}
submitted task: 983aeedd-9a11-47c6-8525-afe7794250c0
18:26:03 PENDING ready= False
18:26:19 PENDING ready= False
18:26:34 PENDING ready= False
18:26:49 PENDING ready= False
18:27:04 PENDING ready= False
18:27:19 PENDING ready= False
18:27:34 PENDING ready= False
18:27:49 PENDING ready= False
18:28:05 PENDING ready= False
18:28:20 PENDING ready= False
18:28:35 PENDING ready= False
18:28:50 PENDING ready= False
18:29:05 PENDING ready= False
18:29:20 PENDING ready= False
18:29:35 PENDING ready= False
18:29:51 PENDING ready= False
18:30:06 PENDING re

: 

## Result Contours

In [14]:
if "result_geojson" not in globals():
    result_geojson = load_json(OUTPUTS / f"{OUTPUT_STEM}.geojson")

features = result_geojson.get("features", [])
print("result contour features:", len(features))
print("idiso distribution:", Counter((feature.get("properties") or {}).get("idiso") for feature in features))
show_geojson(result_geojson, "result", f"NM5 full result contours / {OUTPUT_STEM}.geojson", crs_label="EPSG:4326")

result contour features: 264
idiso distribution: Counter({7: 65, 6: 56, 5: 41, 4: 34, 3: 23, 2: 19, 0: 14, 1: 12})


## Optional: Stop The Stack

Run this only when you are done inspecting results.

In [15]:
STOP_STACK = False
if STOP_STACK:
    run_compose("down")